# ABLH WCT sensitivity study

This notebook estimates the **atmospheric boundary layer height (ABLH)** with two methods:

- **WCT** (`wct`): detects strong vertical gradients in the backscatter signal.
- **Temporal variance** (`temporal_variance`): detects temporal variability associated with the boundary layer.

The main idea is to keep the role of each data source clear:

- **Raw data**: used for the ABLH calculation because it preserves the original temporal resolution.
- **Cloudnet data**: used mainly as a noise mask, because its filtering is useful even though its 5-minute smoothing can be problematic for methods that depend on fine temporal variability.

Run the cells in order. The final section runs a WCT parameter sensitivity study for one date.

## 1. Imports

This cell loads the required libraries. If an import fails, it usually means that the active Python/Jupyter environment is missing a package.

In [ ]:
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from lidarpy.retrieval.ablh import detect_ablh

## 2. Function Definitions

These functions do four things:

1. Open NetCDF files.
2. Prepare the backscatter variable with dimensions `(time, range)`.
3. Apply the Cloudnet mask to the raw data.
4. Run the two ABLH methods and save results/figures.

For normal use, you should not need to edit these functions. Usually, only the parameters in section 3 need to change.

In [ ]:
METHODS = ("wct", "temporal_variance")


def open_netcdf_dataset(input_path: Path) -> xr.Dataset:
    """Open a NetCDF-like file with the first backend that can read it."""
    errors = []
    for engine in (None, "netcdf4", "h5netcdf", "scipy"):
        try:
            if engine is None:
                return xr.open_dataset(input_path)
            return xr.open_dataset(input_path, engine=engine)
        except Exception as exc:
            engine_name = engine or "auto"
            errors.append(f"{engine_name}: {type(exc).__name__}: {exc}")

    joined_errors = "\n".join(errors)
    raise OSError(f"Could not open {input_path} with available xarray backends:\n{joined_errors}")


def load_backscatter_for_ablh(
    input_path: Path,
    *,
    variable: str,
    time_frequency: str | None = None,
) -> xr.Dataset:
    """Load one backscatter variable and return it as an ABLH-ready Dataset.

    Parameters
    ----------
    input_path
        NetCDF file to open.
    variable
        Name of the variable to use. For example, raw files often use
        ``beta_raw`` and Cloudnet files often use ``beta``.
    time_frequency
        Optional resampling frequency, e.g. ``"5min"``. Use ``None`` to keep
        the original temporal resolution.
    """
    input_path = Path(input_path).expanduser()
    if not input_path.exists():
        raise FileNotFoundError(f"Input file not found: {input_path}")

    dataset = open_netcdf_dataset(input_path)
    if variable not in dataset:
        available = ", ".join(dataset.data_vars)
        raise KeyError(
            f"Variable {variable!r} not found in {input_path.name}. "
            f"Available variables: {available}"
        )

    dataset = dataset.sortby("time")

    # Some files can contain duplicated timestamps. Grouping by time gives a
    # clean, unique time axis before optional resampling/reindexing.
    if "time" in dataset.indexes and not dataset.indexes["time"].is_unique:
        dataset = dataset.groupby("time").median()

    backscatter = dataset[variable].transpose("time", "range")
    if time_frequency is not None:
        backscatter = backscatter.resample(time=time_frequency).median(skipna=True)

    output = xr.Dataset({variable: backscatter})
    if "height" in dataset:
        output["height"] = dataset["height"]
    output.attrs.update(dataset.attrs)
    return output


def apply_cloudnet_noise_mask(
    raw_dataset: xr.Dataset,
    cloudnet_dataset: xr.Dataset,
    *,
    raw_variable: str = "beta_raw",
    cloudnet_variable: str = "beta",
) -> xr.Dataset:
    """Mask raw data where Cloudnet has no valid signal.

    Cloudnet has a good noise filter. We use its NaN pattern as a quality mask,
    but we keep the raw values and raw time resolution for the actual ABLH
    calculation.
    """
    masked = raw_dataset.copy()
    cloudnet_nan_mask = np.isnan(cloudnet_dataset[cloudnet_variable])

    # Cloudnet and raw files may not share exactly the same time/range grid.
    # Nearest-neighbour reindexing transfers the Cloudnet mask onto the raw grid.
    cloudnet_nan_mask = cloudnet_nan_mask.reindex_like(
        masked[raw_variable],
        method="nearest",
    )
    masked[raw_variable] = masked[raw_variable].where(~cloudnet_nan_mask)
    return masked


def _log10_positive(values: np.ndarray) -> np.ndarray:
    """Return log10(values), preserving NaNs and avoiding log of non-positive values."""
    positive = np.where(values > 0, values, np.nan)
    finite_positive = np.isfinite(positive)

    if not finite_positive.any():
        return np.full_like(values, np.nan, dtype=float)

    floor = np.nanpercentile(positive[finite_positive], 1)
    if not np.isfinite(floor) or floor <= 0:
        floor = np.nanmin(positive[finite_positive])

    return np.log10(np.maximum(positive, floor))


def plot_ablh_quicklook(
    *,
    dataset: xr.Dataset,
    ablh_results: dict[str, xr.Dataset],
    variable: str,
    min_range: float,
    max_range: float,
    output_path: Path,
) -> None:
    """Plot backscatter plus the ABLH retrieved by each method."""
    backscatter = dataset[variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)

    plot_signal = _log10_positive(backscatter.values[:, range_mask])
    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="0.72", alpha=1.0)

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    ax.set_facecolor("0.72")
    mesh = ax.pcolormesh(
        backscatter["time"].values,
        ranges[range_mask] / 1000.0,
        np.ma.masked_invalid(plot_signal.T),
        shading="auto",
        cmap=cmap,
    )

    colors = {"wct": "white", "temporal_variance": "tab:red"}
    labels = {"wct": "ABLH WCT", "temporal_variance": "ABLH temporal variance"}

    for method, result in ablh_results.items():
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=colors.get(method, "tab:orange"),
            lw=2.0 if method == "wct" else 1.6,
            label=labels.get(method, method),
        )

    title = dataset.attrs.get("title", "Ceilometer")
    ax.set_title(f"{title}: ABLH from {variable}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({variable})")

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output_path, dpi=160)
    plt.close(fig)


def find_file_for_date(directory: Path, date: str, *, pattern: str = "*.nc") -> Path:
    """Find one NetCDF file for a date.

    ``date`` may be written as ``YYYY-MM-DD`` or ``YYYYMMDD``. The search is
    recursive, so it also looks inside subfolders.
    """
    directory = Path(directory).expanduser()
    date_token = date.replace("-", "")
    matches = sorted(directory.rglob(f"*{date_token}*{pattern.replace('*', '')}"))

    if not matches:
        raise FileNotFoundError(f"No file found for date {date} in {directory}")
    if len(matches) > 1:
        print(f"WARNING: several files found for {date}; using the first one:")
        for match in matches[:5]:
            print(f"  - {match}")
        if len(matches) > 5:
            print(f"  ... and {len(matches) - 5} more")

    return matches[0]


def run_ceilometer_ablh_workbench(
    *,
    raw_path: Path,
    cloudnet_path: Path,
    output_dir: Path,
    raw_variable: str = "beta_raw",
    cloudnet_variable: str = "beta",
    time_frequency: str | None = None,
    min_range: float = 500.0,
    max_range: float = 4000.0,
    wct_threshold: float = 0.05,
    wct_width: float = 300.0,
    temporal_threshold: float = 1e-5,
    temporal_window_minutes: float = 10.0,
    apply_cloudnet_mask: bool = True,
    run_label: str | None = None,
) -> tuple[dict[str, Path], Path]:
    """Run WCT and temporal-variance ABLH detection for one day.

    The ABLH is calculated from the raw file. If ``apply_cloudnet_mask=True``,
    Cloudnet is only used to remove noisy raw pixels.
    """
    raw_path = Path(raw_path).expanduser()
    cloudnet_path = Path(cloudnet_path).expanduser()
    output_dir = Path(output_dir).expanduser()
    output_dir.mkdir(parents=True, exist_ok=True)

    label = run_label or raw_path.stem
    frequency_label = time_frequency or "native"
    range_label = f"r{min_range:g}-{max_range:g}m"

    print(f"Loading raw data: {raw_path.name}")
    raw_dataset = load_backscatter_for_ablh(
        raw_path,
        variable=raw_variable,
        time_frequency=time_frequency,
    )

    if apply_cloudnet_mask:
        print(f"Loading Cloudnet mask: {cloudnet_path.name}")
        cloudnet_dataset = load_backscatter_for_ablh(
            cloudnet_path,
            variable=cloudnet_variable,
            time_frequency=None,
        )
        working_dataset = apply_cloudnet_noise_mask(
            raw_dataset,
            cloudnet_dataset,
            raw_variable=raw_variable,
            cloudnet_variable=cloudnet_variable,
        )
    else:
        working_dataset = raw_dataset

    if not np.isfinite(working_dataset[raw_variable]).any():
        raise ValueError(
            f"No finite values remain in {raw_variable!r}. "
            "Check the input files and the Cloudnet mask."
        )

    method_options = {
        "wct": {
            "threshold": wct_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
        "temporal_variance": {
            "threshold": temporal_threshold,
            "wct_width": wct_width,
            "time_window_minutes": temporal_window_minutes,
        },
    }

    output_paths: dict[str, Path] = {}
    results: dict[str, xr.Dataset] = {}

    for method in METHODS:
        print(f"Running ABLH method: {method}")
        ablh_result = detect_ablh(
            working_dataset,
            input_kind="cloudnet",
            variable=raw_variable,
            method=method,
            min_range=min_range,
            max_range=max_range,
            **method_options[method],
        )

        ablh_result.attrs.update(
            {
                "source_raw_file": str(raw_path),
                "source_cloudnet_file": str(cloudnet_path),
                "raw_variable": raw_variable,
                "cloudnet_variable_used_as_mask": cloudnet_variable if apply_cloudnet_mask else "not_used",
                "time_frequency": frequency_label,
                "min_range_m": float(min_range),
                "max_range_m": float(max_range),
                "wct_threshold": float(wct_threshold),
                "wct_width_m": float(wct_width),
                "temporal_threshold": float(temporal_threshold),
                "temporal_window_minutes": float(temporal_window_minutes),
            }
        )

        output_path = output_dir / f"{label}_{method}_{raw_variable}_{frequency_label}_{range_label}_ablh.nc"
        ablh_result.to_netcdf(output_path)
        output_paths[method] = output_path
        results[method] = ablh_result

    quicklook_path = output_dir / f"quicklook_{label}_{raw_variable}_{frequency_label}_{range_label}_ablh.png"
    plot_ablh_quicklook(
        dataset=working_dataset,
        ablh_results=results,
        variable=raw_variable,
        min_range=min_range,
        max_range=max_range,
        output_path=quicklook_path,
    )

    print("Done.")
    return output_paths, quicklook_path

## 3. General Variables and Parameters

This is the section you normally edit.

Adjust `main_dir`, `raw_dir`, `cloudnet_dir`, and `target_date` to match your data. If the raw and Cloudnet files are in the same folder, you can set `raw_dir = main_dir` and `cloudnet_dir = main_dir`.

Tip: use this notebook when you want to inspect or tune one date before running the batch notebook.

In [ ]:
# Main project folder.
main_dir = Path(r"C:\Users\usuario\Documents\github\abl_detection")

# Folders where files are searched recursively.
raw_dir = main_dir / "data" / "raw"
cloudnet_dir = main_dir / "data" / "cloudnet"

# Folder where ABLH NetCDF files and quicklook figures are saved.
output_dir = main_dir / "ablh_outputs"

# Date to process. Accepted formats are YYYY-MM-DD or YYYYMMDD.
target_date = "2023-08-02"

# Variables inside the NetCDF files.
raw_variable = "beta_raw"
cloudnet_variable = "beta"

# Vertical range used to search for the ABLH, in metres above the instrument.
min_range = 500.0
max_range = 4000.0

# Keep None to preserve the original raw temporal resolution.
# Alternative example: "5min", but that smooths/aggregates the temporal signal.
time_frequency = None

# WCT method parameters.
wct_threshold = 0.05
wct_width = 300.0

# temporal_variance method parameters.
temporal_threshold = 1e-5
temporal_window_minutes = 10.0

# If True, use Cloudnet as a noise mask on top of the raw data.
apply_cloudnet_mask = True

### Quick Path Check

This cell does not process anything yet. It only helps you check whether the configured folders exist.

If any path prints `False`, review the path before running the final processing loop.

In [ ]:
print(f"main_dir exists:     {main_dir.exists()} -> {main_dir}")
print(f"raw_dir exists:      {raw_dir.exists()} -> {raw_dir}")
print(f"cloudnet_dir exists: {cloudnet_dir.exists()} -> {cloudnet_dir}")
print(f"output_dir:          {output_dir}")

## 4. Single-Date Processing

This cell processes the date configured in `target_date`.

It finds the matching raw and Cloudnet files, runs both ABLH methods, and saves the NetCDF outputs plus a quicklook figure.

In [ ]:
print("=" * 80)
print(f"Processing date: {target_date}")

raw_path = find_file_for_date(raw_dir, target_date)
cloudnet_path = find_file_for_date(cloudnet_dir, target_date)

print(f"Raw file:      {raw_path}")
print(f"Cloudnet file: {cloudnet_path}")

date_label = target_date.replace("-", "")
ablh_paths, quicklook_path = run_ceilometer_ablh_workbench(
    raw_path=raw_path,
    cloudnet_path=cloudnet_path,
    output_dir=output_dir,
    raw_variable=raw_variable,
    cloudnet_variable=cloudnet_variable,
    time_frequency=time_frequency,
    min_range=min_range,
    max_range=max_range,
    wct_threshold=wct_threshold,
    wct_width=wct_width,
    temporal_threshold=temporal_threshold,
    temporal_window_minutes=temporal_window_minutes,
    apply_cloudnet_mask=apply_cloudnet_mask,
    run_label=date_label,
)

single_output = {
    "target_date": target_date,
    "ablh_paths": ablh_paths,
    "quicklook_path": quicklook_path,
}

single_output

## 5. WCT Sensitivity Study

This section tests a compact grid of WCT parameter combinations for the configured `target_date`.

The goal is not to select the most visually smooth line blindly, but to compare candidates using simple diagnostics:

- `valid_fraction`: fraction of time steps with a finite ABLH estimate.
- `jump_count`: number of large jumps between consecutive estimates.
- `median_abs_step_m`: median absolute step between consecutive estimates.
- `edge_fraction`: fraction of estimates pinned close to `min_range` or `max_range`.
- `score`: lower is better; it penalizes jumps, roughness, edge-pinning, and missing values.

The scan uses `wct_sensitivity_time_frequency` to keep the runtime manageable. Use the ranking table together with the overlay plot. The best physical choice is still the line that follows the boundary layer, not necessarily the lowest score.

In [ ]:
# WCT sensitivity parameter grid.
wct_sensitivity_thresholds = [0.04, 0.05, 0.06, 0.08, 0.10]
wct_sensitivity_widths = [250.0, 300.0, 400.0]

# Use a coarser time grid for the parameter scan so the study stays interactive.
wct_sensitivity_time_frequency = "10min"

# Diagnostics.
wct_jump_limit_m = 250.0
wct_edge_margin_m = 75.0

# Number of top-ranked candidates shown on the overlay plot.
wct_top_n_to_plot = 6

In [ ]:
def build_wct_sensitivity_dataset() -> tuple[xr.Dataset, Path, Path]:
    """Load and mask the configured single-date data without running ABLH."""
    raw_path = find_file_for_date(raw_dir, target_date)
    cloudnet_path = find_file_for_date(cloudnet_dir, target_date)

    raw_dataset = load_backscatter_for_ablh(
        raw_path,
        variable=raw_variable,
        time_frequency=wct_sensitivity_time_frequency,
    )

    if apply_cloudnet_mask:
        cloudnet_dataset = load_backscatter_for_ablh(
            cloudnet_path,
            variable=cloudnet_variable,
            time_frequency=None,
        )
        working_dataset = apply_cloudnet_noise_mask(
            raw_dataset,
            cloudnet_dataset,
            raw_variable=raw_variable,
            cloudnet_variable=cloudnet_variable,
        )
    else:
        working_dataset = raw_dataset

    if not np.isfinite(working_dataset[raw_variable]).any():
        raise ValueError(
            f"No finite values remain in {raw_variable!r}. Check the input files and mask."
        )

    return working_dataset, raw_path, cloudnet_path


def summarize_ablh_candidate(
    ablh_result: xr.Dataset,
    *,
    threshold: float,
    wct_width: float,
) -> dict[str, float]:
    """Calculate simple stability diagnostics for one WCT candidate."""
    values = np.asarray(ablh_result["ablh"].values, dtype=float)
    finite = np.isfinite(values)
    finite_values = values[finite]

    if finite_values.size == 0:
        return {
            "threshold": threshold,
            "wct_width": wct_width,
            "valid_fraction": 0.0,
            "median_ablh_m": np.nan,
            "p05_ablh_m": np.nan,
            "p95_ablh_m": np.nan,
            "median_abs_step_m": np.nan,
            "jump_count": np.nan,
            "jump_rate": np.nan,
            "edge_fraction": np.nan,
            "score": np.inf,
        }

    steps = np.abs(np.diff(finite_values))
    jump_count = int(np.sum(steps > wct_jump_limit_m)) if steps.size else 0
    jump_rate = jump_count / max(steps.size, 1)
    median_abs_step = float(np.nanmedian(steps)) if steps.size else 0.0
    edge_fraction = float(
        np.mean(
            (finite_values <= min_range + wct_edge_margin_m)
            | (finite_values >= max_range - wct_edge_margin_m)
        )
    )
    valid_fraction = float(np.mean(finite))

    score = (
        3.0 * jump_rate
        + median_abs_step / 500.0
        + 2.0 * edge_fraction
        + (1.0 - valid_fraction)
    )

    return {
        "threshold": threshold,
        "wct_width": wct_width,
        "valid_fraction": valid_fraction,
        "median_ablh_m": float(np.nanmedian(finite_values)),
        "p05_ablh_m": float(np.nanpercentile(finite_values, 5)),
        "p95_ablh_m": float(np.nanpercentile(finite_values, 95)),
        "median_abs_step_m": median_abs_step,
        "jump_count": jump_count,
        "jump_rate": float(jump_rate),
        "edge_fraction": edge_fraction,
        "score": float(score),
    }

In [ ]:
sensitivity_records = []
wct_sensitivity_results = {}

working_dataset, sensitivity_raw_path, sensitivity_cloudnet_path = build_wct_sensitivity_dataset()

print(f"Raw file:      {sensitivity_raw_path}")
print(f"Cloudnet file: {sensitivity_cloudnet_path}")
print(f"Testing {len(wct_sensitivity_thresholds) * len(wct_sensitivity_widths)} WCT combinations...")

for threshold in wct_sensitivity_thresholds:
    for width in wct_sensitivity_widths:
        result = detect_ablh(
            working_dataset,
            input_kind="cloudnet",
            variable=raw_variable,
            method="wct",
            min_range=min_range,
            max_range=max_range,
            threshold=threshold,
            wct_width=width,
        )
        key = (threshold, width)
        wct_sensitivity_results[key] = result
        sensitivity_records.append(
            summarize_ablh_candidate(
                result,
                threshold=threshold,
                wct_width=width,
            )
        )

sensitivity_df = (
    pd.DataFrame(sensitivity_records)
    .sort_values(["score", "jump_count", "median_abs_step_m"])
    .reset_index(drop=True)
)

wct_sensitivity_output_dir = Path(output_dir)
wct_sensitivity_output_dir.mkdir(parents=True, exist_ok=True)
wct_sensitivity_date_label = target_date.replace("-", "")
wct_sensitivity_table_path = (
    wct_sensitivity_output_dir / f"wct_sensitivity_table_{wct_sensitivity_date_label}.csv"
)
sensitivity_df.to_csv(wct_sensitivity_table_path, index=False)
print(f"Saved sensitivity table: {wct_sensitivity_table_path}")

sensitivity_df.head(12)

In [ ]:
ranked_wct_sensitivity = sensitivity_df.copy()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

heatmap_specs = [
    ("score", "Score", "viridis_r"),
    ("jump_count", f"Jumps > {wct_jump_limit_m:g} m", "magma_r"),
    ("median_abs_step_m", "Median absolute step [m]", "magma_r"),
]

for ax, (column, title, cmap_name) in zip(axes, heatmap_specs):
    pivot = ranked_wct_sensitivity.pivot(
        index="wct_width",
        columns="threshold",
        values=column,
    )
    image = ax.imshow(
        pivot.values,
        origin="lower",
        aspect="auto",
        cmap=cmap_name,
    )
    ax.set_title(title)
    ax.set_xlabel("WCT threshold")
    ax.set_ylabel("WCT width [m]")
    ax.set_xticks(np.arange(len(pivot.columns)))
    ax.set_xticklabels([f"{value:g}" for value in pivot.columns], rotation=45)
    ax.set_yticks(np.arange(len(pivot.index)))
    ax.set_yticklabels([f"{value:g}" for value in pivot.index])
    fig.colorbar(image, ax=ax)

fig.suptitle(f"WCT sensitivity diagnostics for {target_date}")

wct_sensitivity_heatmap_path = (
    wct_sensitivity_output_dir / f"wct_sensitivity_heatmaps_{wct_sensitivity_date_label}.png"
)
fig.savefig(wct_sensitivity_heatmap_path, dpi=180)
print(f"Saved sensitivity heatmap figure: {wct_sensitivity_heatmap_path}")
plt.show()

In [ ]:
def plot_wct_sensitivity_overlay(
    *,
    dataset: xr.Dataset,
    ranked_candidates,
    results: dict[tuple[float, float], xr.Dataset],
    top_n: int,
) -> None:
    """Plot the top-ranked WCT candidates over the backscatter quicklook."""
    backscatter = dataset[raw_variable].transpose("time", "range")
    ranges = backscatter["range"].values.astype(float)
    range_mask = (ranges >= min_range) & (ranges <= max_range)
    plot_signal = _log10_positive(backscatter.values[:, range_mask])

    cmap = plt.get_cmap("viridis").copy()
    cmap.set_bad(color="0.72", alpha=1.0)

    fig, ax = plt.subplots(figsize=(13, 5), constrained_layout=True)
    ax.set_facecolor("0.72")
    mesh = ax.pcolormesh(
        backscatter["time"].values,
        ranges[range_mask] / 1000.0,
        np.ma.masked_invalid(plot_signal.T),
        shading="auto",
        cmap=cmap,
    )

    candidate_rows = ranked_candidates.head(top_n)
    line_colors = plt.cm.tab10(np.linspace(0, 1, len(candidate_rows)))
    for color, (_, row) in zip(line_colors, candidate_rows.iterrows()):
        key = (float(row["threshold"]), float(row["wct_width"]))
        result = results[key]
        label = (
            f"thr={row['threshold']:g}, width={row['wct_width']:g} m, "
            f"score={row['score']:.2f}"
        )
        ax.plot(
            result["time"].values,
            result["ablh"].values / 1000.0,
            color=color,
            lw=1.8,
            label=label,
        )

    ax.set_title(f"Top {top_n} WCT candidates for {target_date}")
    ax.set_xlabel("UTC time")
    ax.set_ylabel("Range above instrument [km]")
    ax.set_ylim(min_range / 1000.0, max_range / 1000.0)
    ax.grid(True, alpha=0.25)
    ax.legend(loc="upper right", fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

    cbar = fig.colorbar(mesh, ax=ax)
    cbar.set_label(f"log10({raw_variable})")

    wct_sensitivity_overlay_path = (
        wct_sensitivity_output_dir / f"wct_sensitivity_overlay_{wct_sensitivity_date_label}.png"
    )
    fig.savefig(wct_sensitivity_overlay_path, dpi=180)
    print(f"Saved sensitivity overlay figure: {wct_sensitivity_overlay_path}")
    plt.show()


plot_wct_sensitivity_overlay(
    dataset=working_dataset,
    ranked_candidates=ranked_wct_sensitivity,
    results=wct_sensitivity_results,
    top_n=wct_top_n_to_plot,
)

### Apply a Selected WCT Candidate

After inspecting the table and overlay, copy one candidate into the main parameter cell:

```python
wct_threshold = ...
wct_width = ...
```

Then rerun the single-date processing cell above to generate the final quicklook and NetCDF outputs.